In [8]:
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession

In [9]:
builder = (SparkSession.builder
           .appName("create-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")   
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [10]:
get_ipython().run_line_magic('load_ext', 'sparksql_magic')
get_ipython().run_line_magic('config', 'SparkSql.limit=20')

The sparksql_magic extension is already loaded. To reload it, use:
  %reload_ext sparksql_magic


In [11]:
%%sparksql
DROP TABLE IF EXISTS default.netflix_titles;


In [12]:
%%sparksql
CREATE OR REPLACE TABLE default.netflix_titles (
    show_id STRING,
    type STRING,
    title STRING,
    director STRING,
    cast STRING,
    country STRING,
    date_added STRING,
    release_year STRING,
    rating STRING,
    duration STRING,
    listed_in STRING,
    description STRING 
) USING DELTA LOCATION '../../data/delta_lake/netflix_titles';

24/08/22 14:58:27 ERROR Utils: Aborting task
org.apache.spark.sql.delta.DeltaAnalysisException: Cannot create table ('`default`.`netflix_titles`'). The associated location ('file:/home/jovyan/work/notebooks/data/delta_lake/netflix_titles') is not empty and also not a Delta table.
	at org.apache.spark.sql.delta.DeltaErrorsBase.createTableWithNonEmptyLocation(DeltaErrors.scala:2489)
	at org.apache.spark.sql.delta.DeltaErrorsBase.createTableWithNonEmptyLocation$(DeltaErrors.scala:2488)
	at org.apache.spark.sql.delta.DeltaErrors$.createTableWithNonEmptyLocation(DeltaErrors.scala:2794)
	at org.apache.spark.sql.delta.commands.CreateDeltaTableCommand.assertPathEmpty(CreateDeltaTableCommand.scala:265)
	at org.apache.spark.sql.delta.commands.CreateDeltaTableCommand.createTransactionLogOrVerify$1(CreateDeltaTableCommand.scala:186)
	at org.apache.spark.sql.delta.commands.CreateDeltaTableCommand.$anonfun$run$2(CreateDeltaTableCommand.scala:211)
	at org.apache.spark.sql.delta.metering.DeltaLogging.

AnalysisException: Cannot create table ('`default`.`netflix_titles`'). The associated location ('file:/home/jovyan/work/notebooks/data/delta_lake/netflix_titles') is not empty and also not a Delta table.

In [7]:
import os
print(os.getcwd())  # Prints the current working directory
print(os.path.exists("../../data/netflix_titles.csv"))  # Checks if the file exists

/home/jovyan/work/notebooks/Chapter03
True


In [ ]:
import os

# Specify the directory path
directory_path = '../../data'

# List and print all files in the specified directory
files = os.listdir(directory_path)
print("Files in the directory:", files)


In [ ]:
df = (spark.read
      .format("csv")
      .option("header", "true")
      .load("../../data/netflix_titles.csv"))


In [ ]:
df.printSchema()

In [ ]:
spark.sql("DROP TABLE IF EXISTS default.netflix_titles")

In [ ]:
# Define the path within the delta-tables volume
delta_table_path = "../../data/delta_lake/netflix_titles"

# Write the DataFrame to a Delta table
df.write.format("delta") \
    .mode("overwrite") \
    .option("path", delta_table_path) \
    .saveAsTable("default.netflix_titles")


In [ ]:
%%sparksql 
SELECT * FROM default.netflix_titles LIMIT 3;

In [ ]:
spark.stop()